In [4]:
:dep smartcore = { version = "0.3.2", features = ["ndarray-bindings", "serde"] }
:dep bincode = "1.3"
:dep csv = "1.1"
:dep ndarray = "0.15" 
:dep linfa = { version = "0.7", features = ["default"] }
:dep rand = "0.8"
:dep rand_chacha = "0.3"
:dep plotters = "0.3"

In [5]:
use smartcore::{neighbors::knn_classifier::*,linalg::basic::matrix::DenseMatrix,
                naive_bayes::gaussian::GaussianNB,tree::decision_tree_classifier::DecisionTreeClassifier,
                metrics::distance::Distances,tree::decision_tree_classifier::DecisionTreeClassifierParameters,
                tree::decision_tree_classifier::SplitCriterion};
use csv;
use linfa::{Dataset};
use ndarray::{Array1, Array2, ArrayView1, ArrayView2, Axis, Ix1, array,s};
use rand::SeedableRng;
use rand_chacha::ChaCha12Rng;

In [6]:
let path= "ph-data.csv";
let mut reader =csv::Reader::from_path(path).unwrap();
let mut records: Array2<f64> = Array2::zeros((0, 3));  //X
let mut targets: Array1<f64>= Array1::zeros(0); //Y
let mut rng = ChaCha12Rng::seed_from_u64(5);

In [7]:
for data in reader.records(){
        let rec = data.unwrap();
        let array = array![[
                            rec[0].parse::<f64>()
                                  .expect("pupupu"),

                            rec[1].parse::<f64>()
                                  .expect("pupupu"),

                            rec[2].parse::<f64>()
                                  .expect("pupupu")
                            ]];
        let _  = records.append(Axis(0), array.view());
        if rec[3].parse::<i32>().expect("pupupu")>9{
            let _ = targets.append(Axis(0),array![1.0].view());
        }
        else if rec[3].parse::<i32>().expect("pupupu")<6{
            let _ = targets.append(Axis(0),array![-1.0].view());
        }
        else{
            let _  = targets.append(Axis(0),array![0.0].view());
        }
    }
let data_set= Dataset::new(records, targets);
let (train_data,test_data) = data_set.shuffle(&mut rng)
                                     .split_with_ratio(0.7);

In [8]:

let x = DenseMatrix::from_2d_vec(
                              &train_data.records().axis_iter(Axis(0))
                                                 .map(|row| row.to_vec())
                                                 .collect::<Vec<Vec<f64>>>()
                                );
let y = train_data.targets()
                .map(|value| *value as i32)
                .to_vec();
let knn=KNNClassifier::fit(&x, &y, Default::default()).unwrap(); //обучение
let y_unsigned = train_data.targets()
                .map(|value| (*value +1.0) as usize)
                .to_vec();
let gnb = GaussianNB::fit(&x, &y_unsigned, Default::default()).unwrap(); //обучение, при этом классы беззнаковые 
let dt = DecisionTreeClassifier::fit(&x, &y, Default::default()).unwrap(); //обучение

In [9]:
let test_x = DenseMatrix::from_2d_vec(
                              &test_data.records().axis_iter(Axis(0))
                                                 .map(|row| row.to_vec())
                                                 .collect::<Vec<Vec<f64>>>()
                                );
let test_y = test_data.targets()
                .map(|value| *value as i32)
                .to_vec();
let knn_res = knn.predict(&test_x).unwrap();
let gnb_res: Vec<i32> =gnb.predict(&test_x).unwrap()
                          .into_iter()
                          .map(|x| (x as i32) -1)
                          .collect();
let dt_res = dt.predict(&test_x).unwrap();

In [10]:
let mut knn_acc = 0.0;
let mut gnb_acc = 0.0;
let mut dt_acc = 0.0;
for i in 0..test_y.len(){
    if knn_res[i] == test_y[i]{
        knn_acc=knn_acc+1.0;
    }
    if gnb_res[i] == test_y[i]{
        gnb_acc=gnb_acc+1.0;
    }
    if dt_res[i] == test_y[i]{
        dt_acc=dt_acc+1.0;
    }
}
knn_acc=knn_acc/ (test_y.len() as f64);
gnb_acc=gnb_acc/ (test_y.len() as f64);
dt_acc=dt_acc/ (test_y.len() as f64);
println!("knn accuracity: {} | gnb accuracity {} | dt accuracity {}",knn_acc,gnb_acc,dt_acc);

knn accuracity: 0.9282051282051282 | gnb accuracity 0.9076923076923077 | dt accuracity 0.9128205128205128


In [11]:
let knn_params = KNNClassifierParameters::default()
                                    .with_k(5)
                                    .with_distance(Distances::manhattan());
let knn=KNNClassifier::fit(&x, &y, knn_params).unwrap(); //обучение
let y_unsigned = train_data.targets()
                .map(|value| (*value +1.0) as usize)
                .to_vec();
let dt_params = DecisionTreeClassifierParameters::default()
                                                  .with_criterion(SplitCriterion::Entropy)
                                                  .with_max_depth(20)
                                                  .with_min_samples_leaf(15)
                                                  .with_min_samples_split(5);
let dt = DecisionTreeClassifier::fit(&x, &y, dt_params).unwrap(); //обучение

In [12]:
let knn_res = knn.predict(&test_x).unwrap();
let gnb_res: Vec<i32> =gnb.predict(&test_x).unwrap()
                          .into_iter()
                          .map(|x| (x as i32) -1)
                          .collect();
let dt_res = dt.predict(&test_x).unwrap();

In [13]:
let mut knn_acc = 0.0;
let mut gnb_acc = 0.0;
let mut dt_acc = 0.0;
for i in 0..test_y.len(){
    if knn_res[i] == test_y[i]{
        knn_acc=knn_acc+1.0;
    }
    if gnb_res[i] == test_y[i]{
        gnb_acc=gnb_acc+1.0;
    }
    if dt_res[i] == test_y[i]{
        dt_acc=dt_acc+1.0;
    }
}
knn_acc=knn_acc/ (test_y.len() as f64);
gnb_acc=gnb_acc/ (test_y.len() as f64);
dt_acc=dt_acc/ (test_y.len() as f64);
println!("knn accuracity: {} | gnb accuracity {} | dt accuracity {}",knn_acc,gnb_acc,dt_acc);

knn accuracity: 0.9230769230769231 | gnb accuracity 0.9076923076923077 | dt accuracity 0.9384615384615385


In [14]:
let data_rec = DenseMatrix::from_2d_vec(
                              &data_set.records().axis_iter(Axis(0))
                                                 .map(|row| row.to_vec())
                                                 .collect::<Vec<Vec<f64>>>()
                                );
let data_targ = data_set.targets()
                .map(|value| *value as i32)
                .to_vec();

In [15]:
let dt_params = DecisionTreeClassifierParameters::default()
                                                  .with_criterion(SplitCriterion::Entropy)
                                                  .with_max_depth(20)
                                                  .with_min_samples_leaf(15)
                                                  .with_min_samples_split(5);
let dt = DecisionTreeClassifier::fit(&data_rec, &data_targ, dt_params).unwrap(); //обучение

In [16]:
let test_rec = DenseMatrix::from_2d_vec(&vec![
                                              vec![111.0,41.0,142.0],
                                              vec![179.0,68.0,198.0],
                                              vec![141.0,250.0,0.0],
                                              vec![255.0,124.0,0.0],
                                              vec![255.0,38.0,0.0],
                                             ]);
let test_targ = vec![-1.0,-1.0,0.0,1.0,1.0];

In [17]:
let dt_res = dt.predict(&test_rec).unwrap();

In [18]:
let mut dt_acc = 0.0;
for i in 0..test_targ.len(){
    if dt_res[i] as f64 == test_targ[i]{
        dt_acc=dt_acc+1.0;
    }
}
dt_acc=dt_acc/ (test_targ.len() as f64);
println!("FINAL dt accuracity 🦀🦀🦀 {}",dt_acc);

FINAL dt accuracity 🦀🦀🦀 1


In [19]:
use std::fs::File;
use std::io::Write;

//сохраняем
let bytes = bincode::serialize(&dt).map_err(|e| e.to_string()).unwrap();
File::create("model.bin").unwrap().write_all(&bytes).unwrap();